# Проект по дисциплине "Теория конечных графов" группа№3

Ладнюк Кира

Гареев Эльдар

Егоров Иван

# 0. Подготовка.

Импортируем необходимые библиотеки

In [ ]:
import os
from collections import defaultdict, deque, Counter
import numpy as np
from random import randint, random, sample, choice
import matplotlib.pyplot as plt
import heapq

Напишем функцию для обработки неориентированных графов. Графы будем хранить в виде словаря: ключами будут номера вершин, записями будут множества с номерами смежных вершин. При возникновении ошибки будет выходить соответствующее сообщение.

In [ ]:
def make_graph(path, type='txt'):
    # Создаем словарь, который по умолчанию будет присваивать ключу пустое множество
    graph = defaultdict(set)
    unique_edges = 0
    all_edges = 0

    try:
        with open(path, 'r') as file:
            if type == 'txt':
                for line in file:
                    # Убираем лишние символы в начале и конце строки
                    line = line.strip()

                    # Пропускаем комментарии
                    if line.startswith('#') or not line:
                        continue

                    u, v = line.split()
                    u = int(u)
                    v = int(v)

                    # Считаем общее количество ребер (включая кратные)
                    all_edges += 1

                    # Если ребро (u, v) ещё не было добавлено в словарь, добавляем (u, v) и (v, u)
                    if v not in graph[u]:

                        graph[u].add(v)
                        graph[v].add(u)

                        # Считаем уникальные ребра
                        unique_edges += 1
            elif type == 'mtx':
                lines = file.readlines()

                # Считаем заголовок и проверим наличие параметра симметричности
                # Файл сохраняет граф как матрицу, если она симметрична - граф неориентированный
                head = lines.pop(0)
                symmetric = 'symmetric' in head

                # Считаем следующую строку, чтобы получить число ребер
                _, _, all_edges = [int(i) for i in lines.pop(0).split()]

                for line in lines:
                    line = line.strip()

                    u, v = line.split()
                    u = int(u)
                    v = int(v)

                    if v not in graph[u]:
                        graph[u].add(v)
                        if symmetric:
                            graph[v].add(u)

                        unique_edges += 1
            elif type == 'csv':
                lines = file.readlines()
                lines.pop(0)
                for line in lines:                    
                    data = list(map(int, line.strip().split(',')))
                    u = data[0]
                    v = data[1]
                    all_edges += 1

                    if v not in graph[u]:
                        graph[u].add(v)
                        graph[v].add(u)

                        unique_edges += 1


    except FileNotFoundError:
        print(f"Ошибка: файл {path} не найден.")
        return None

    except Exception as e:
        print(f"Произошла ошибка при обработке файла: {e}")
        return None

    return  dict(graph), unique_edges, all_edges

Напишем функцию для генерации случайных графов. В качестве аргументов будем передавать желаемое количество графов:

In [ ]:
def generate_graphs(n=0, oriented=False):
    if not n:
        n = randint(5, 15)
    graphs = []

    for _ in range(n):
        # Выбираем число вершин, равномерно распределенное от 5000 до 10000
        v = randint(5000, 10000)
        vertices = [i for i in range(v)]
        # Выбираем количество ребер как максимально возможное, умноженное на коэффициент от 0 до 1
        e = int(((v * (v - 1)) / 2) * random())
        graph = {vertice: set() for vertice in vertices}
        while e:
            u, v = sample(vertices, 2)
            if v not in graph[u]:
                graph[u].add(v)
                if not oriented:
                    graph[v].add(u)
                e -= 1
        graphs.append(graph)
    return graphs

Будем использовать файл WikiVote в качестве примера и анализировать его, как неориентированный граф:

In [ ]:
file = "./vk.csv"
graph, edges, _ = make_graph(file, type='csv')
print(len(graph), edges)

# Анализ структуры сети
## Задание А
### _Часть 1_

Для каждой из сетей определить следующие характеристики:
Число вершин, число рёбер, плотность (отношение числа рёбер к максимально возможному числу рёбер), число компонент слабой связности, долю вершин в максимальной по мощности компоненте слабой связности. Для ориентированных графов определить число компонент сильной связности и долю вершин графа в наибольшей компоненте сильной связности компоненте.

Начнем с числа вершин и ребер:
  - кол-во вершин = кол-во ключей в словаре
  - кол-во ребер =  каждое ребро (A, B) хранится дважды, тогда общее количество рёбер можно получить как сумма мощностей множеств, делённая на 2
  - плотность = отношение числа рёбер к максимально возможному числу рёбер

In [ ]:
num_vertex = len(graph)
max_num_vertex = num_vertex * (num_vertex - 1) // 2
density = edges / max_num_vertex if max_num_vertex > 0 else 0.0

print(f"Кол-во вершин: {num_vertex}, кол-во ребер: {edges}, плотность: {density:.6f}")

todo: сделать визуализацию


Поиск компонент слабой связности. Для этого:
1. Реализуем DFS с помощью стека
2. Запускаем DFS по всем ключам словаря.
3. Функция вернет списков из списков, где каждый элемент - комппонента слабой связности.

In [ ]:
def dfs_stack(graph):

    visited = set()
    components = []

    for u in graph:

        if u not in visited:
            stack = [u]
            component = []

            while stack:
                cur = stack.pop()

                if cur not in visited:
                    visited.add(cur)
                    component.append(cur)

                    for v in graph[cur]:
                        if v not in visited:
                            stack.append(v)

            components.append(component)
    return components


Рассмотрим сколько компонент слабой связности у нас получилось: для этого выведем длину получишегося массива.

In [ ]:
components = dfs_stack(graph)
print(f'Число компонент слабой связности в графе равно {len(components)}')

Найдем долю вершин в максимальной по мощности компоненте слабой связности:
1. Найдем компоненту с максимальным числом вершин
2. Поделим мощность компоненты с максимальным числом вершин на общее количество вершин

In [ ]:
max_size = max(len(c) for c in components)
fraction = max_size / num_vertex
print(f'Доля вершин в компоненте слабой связности равна {(fraction * 100):.2f}%')

### _Часть 2, пункт а_

Для наибольшей компоненты слабой связности оценить значения диаметра сети, 90 процентиля расстояния (геодезического) между вершинами графа. Оценку провести на основании **двойного прохода BFS** (the double sweep): для случайно выбранного узла найти максимально удаленный узел _a_, а затем найти узел _b_, максимально удаленный от _a_. За диаметр принять эксцентриситет вершины _ecc(a) = d(b,a)_

Для решения этого пункта:

1. Найдем наибольшую компоненту слабой связности
2. Используем алгоритм BFS
3. Выберем случайную вершину u, найдем самую удаленную от нее вершину v (первый проход BFS).
4. Найдем самую удаленную от v вершину w (второй проход BFS)

Тогда расстояние d(v, w) будет приближенным диаметром

Функция для нахождения самой большой компоненты слабой связности с помощью подсчета вершин в каждой компоненте. На выход возвращает подграф в виде словаря.

In [ ]:
def subgraph(components, graph):
    comp = max(components, key=len)
    nodes = set(comp)
    ans = {}

    # Переносим в ответ данные о тех ребрах, которые соединяют вершины наибольшей компоненты слабой связности
    for node in comp:
        ans[node] = {i for i in graph.get(node, set()) if i in nodes}

    return ans

In [ ]:
large_comp = subgraph(components, graph)
print(f'Размер самой большой компоненты слабой связности равен {len(large_comp)}')

Реализация алгоритма BFS на очереди. Функция возвращает самую дальнюю вершину от выбранной и расстояния до всех вершин компоненты

In [ ]:
def bfs_far(graph, u):

    dist = {u: 0}
    queue = deque([u])
    u = u
    max_dist = 0

    while queue:

        cur = queue.popleft()

        for v in graph.get(cur, set()):
            if v not in dist:

                dist[v] = dist[cur] + 1
                queue.append(v)
                if dist[v] > max_dist:
                    u = v
                    max_dist = dist[v]

    return u, dist


Выбираем случайную вершину, запускаем из нее BFS. В полученном дереве выбираем наиболее удаленную от выбранной вершины и запускаем повторный поиск из нее. Во втором полученном дереве находим расстояние (в ребрах) до самой дальней вершины и считаем полученную величину приближенным диаметром графа.

In [ ]:
def dbl_swp_diam(graph):
    if not graph:
        return 0

    u = choice(list(graph))
    v, _ = bfs_far(graph, u)
    _, ans = bfs_far(graph, v)

    return max(ans.values())

In [ ]:
diameter = dbl_swp_diam(large_comp)
print(f'Диаметр сети, найденный с помощью двух вершин, равен {diameter}')

### _Часть 2, пункт b_
Вычисление 90-процентиля расстояний между случайными вершинами:
  1. Выбираем 1000 или 500 (по умолчанию) случайных пар вершин
  2. Для первой вершины делаем BFS, реализованный раннее
  3. Для второй проверяем, попадает ли она в дерево BFS для первой вершины. Если да, то добавляем расстояние до нее в список
  4. Сортируем расстояния и находим 90-процентиль.

In [ ]:
def percentile90(graph, n=500):

    nodes = list(graph.keys()) if hasattr(graph, 'keys') else list(graph)

    distances = []

    for _ in range(n):
        # Выберем две случайные вершины
        u, v = sample(nodes, 2)

        # Посчитаем расстояние с помощью BFS
        _, d = bfs_far(graph, u)

        if v in d:
            distances.append(d[v])

    # Используем функцию из библиотеки numpy, чтобы найти n-процентиль выборки
    return int(np.percentile(distances, 90)) if distances else 0

In [ ]:
ans = percentile90(large_comp)

print(f'90-Процентиль расстояний в большой компоненте равен {ans}')

### _Часть 2, пункт c_

Используем Snowball Sampling: будем добавлять к выборке первые n (500) вершин, которые посетим из некоторых начальных. Будем присоединять к компоненте вершины с помощью поиска в ширину, начиная из некоторых стартовых. Сформируем из выбранных вершин и ребер, соединяющих их, компоненту, а затем получим 90-процентиль расстояний в этой компоненте.

In [ ]:
np.random.seed(26)
def snowball(graph, n=500):
    if not graph: return {}

    # Выбираем случаные вершины, с которых начнем
    s = sample(list(graph), min(3, len(graph)))
    visited, queue = set(s), deque(s)

    # Будем продолжать, пока не наберем n соседей или пока не иссякнет очередь
    while queue and len(visited) < n:
        c = queue.popleft()
        for i in graph.get(c, []):
            if i not in visited and len(visited) < n:
                visited.add(i)
                queue.append(i)

    sg = {node: set() for node in visited}
    for node in visited:
        sg[node] = {nb for nb in graph.get(node, []) if nb in visited}

    return sg

def snowball_analysis(graph, n=500):
    subgraph = snowball(graph, n)
    diameter = dbl_swp_diam(subgraph)
    percentile = percentile90(subgraph, n=100)
    return diameter, percentile

In [ ]:
sn_dim, sn_per = snowball_analysis(large_comp)
print(f'90-Процентиль расстояний в снежном коме равен {sn_per}, диаметр равен {sn_dim}')

### _Часть 3_
Посчитаем количество треугольников: для каждой вершины графа будем перебирать ее соседей и находить уникальные пары, соединенные ребром. Для того, чтобы не повторяться в подсчетах, будем идти по возрастанию номеров вершин.

In [ ]:
def triangles(graph):

    triangles = 0

    for u in graph:
        current = graph[u]
        for v in current:
            # Анализируем соседей
            if v > u:
                for w in current & graph[v]:
                    if w > v:
                        triangles += 1

    return triangles

In [ ]:
tri_number = triangles(graph)
print(f'Количество полных подграфов на 3 вершинах равно {tri_number}')

### _Часть 4_


Вычислим средний кластерный коэффициент с помощью формулы, указанной в задании.

In [ ]:
def node_clust(graph, n):
    nodes = graph.get(n, set())
    neighbours = len(nodes)
    if neighbours < 2: return 0.0

    max = neighbours * (neighbours - 1) / 2
    unique = 0

    for u in nodes:
        for v in nodes:
            if u > v and v in graph.get(u, set()):
                unique += 1

    return unique / max

def avg_clust(graph):

    ans = 0.0
    count = 0

    for node in graph:
        coeff = node_clust(graph, node)
        ans += coeff
        count += 1

    return ans / count

In [ ]:
print(f'Средний кластерный коэффициент в компоненте составляет: {avg_clust(large_comp):.4f}')
print(f'Средний кластерный коэффициент в графе составляет: {avg_clust(graph):.4f}')

In [ ]:
def gcc(g):
    total, count = 0, 0

    for u in g:
        n = g[u]
        k = len(n)

        if k < 2: 
            continue
        total += k * (k - 1) / 2

        for v in n:
            for w in n:
                if v > w and w in g.get(v,set()):
                    count += 1

    return count / total if total else 0.0

In [ ]:
print(f'Глобальный кластерный коэффициент в графе составляет: {avg_clust(graph):.4f}')

In [ ]:
def avg_clust(g, comp):
    if not comp: return 0.0

    total = 0.0
    nodes = set(comp)

    for u in comp:
        nb = [v for v in g.get(u, set()) if v in nodes]
        k = len(nb)
        if k < 2: continue

        edges = 0
        for i in range(k):
            for j in range(i+1, k):
                if nb[j] in g.get(nb[i], set()):
                    edges += 1

        total += (2 * edges) / (k * (k - 1))

    return total / len(comp)

### _Часть 5_
Найдем минимальную, максимальную и среднюю степени вершин в графе, посчитав и упорядочив степени всех вершин в графе

In [ ]:
def calculate_node_degrees(graph):
    degrees = [len(u) for u in graph.values()]

    return min(degrees),max(degrees), sum(degrees) / len(degrees)

In [ ]:
min_deg, max_deg, avg_deg = calculate_node_degrees(graph)
print(f'Минимальная степень вершины в графе равна {min_deg}')
print(f'Максимальная степень вершины в графе равна {max_deg}')
print(f'Средняя степень вершины в графе равна {avg_deg:.4f}')

Посчитаем степени для вершин, затем посчитаем, сколько вершин имеют одинаковые степени и построим распределение. Отобразим на линейном и логарифмическом графике.

In [ ]:
def plot_degree_distribution(graph, log_log=True):
    degrees = [len(neighbors) for neighbors in graph.values()]
    degree_counts = Counter(degrees)

    k_values = sorted(degree_counts.keys())
    p_k = [degree_counts[k]/len(graph) for k in k_values]

    # Обычный масштаб
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.bar(k_values, p_k, width=0.9, alpha=0.85)
    plt.title('Распределение степеней (линейная шкала)')
    plt.xlabel('Степень k')
    plt.ylabel('Вероятность P(k)')
    plt.grid(True, linestyle='--', alpha=0.6)

    # Log-log масштаб
    if log_log:
        plt.subplot(1, 2, 2)
        plt.loglog(k_values, p_k, 'bo', markersize=5, alpha=0.4)
        plt.title('Распределение степеней (log-log шкала)')
        plt.xlabel('log(k)')
        plt.ylabel('log(P(k))')
        plt.grid(True, which="both", linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

    # Возвращаем функцию вероятности в виде распределения
    return dict(zip(k_values, p_k))

In [ ]:
degree_dist = plot_degree_distribution(graph)

## Задание B

Для каждой из сетей исследовать, как меняется доля вершин в наибольшей компоненте слабой связности, если из сети удаляется:
1. Cлучайным образом x% узлов
2. X% узлов наибольшей степени.

Проанализируем устойчивость к случайному удалению узлов в функции. На вход будем подавать граф в привычном нам виде, список процентов для удаления и число испытаний. На выходе будем возвращать словарь, где ключами будут проценты удаления, а значениями - средние доли вершин в наибольшей компоненте.

### _Часть 1_

In [ ]:
def analyze_component_resilience(graph, x_values, num_trials=5):
    components = dfs_stack(graph)
    lcc = subgraph(components, graph)
    results = defaultdict(list)
    lcc_nodes = set(lcc.keys())
    original_size = len(lcc)

    for x in x_values:
        for _ in range(num_trials):
            # Создаем копию только для узлов LCC
            current_lcc = {node: set(neighbors) for node, neighbors in lcc.items()}

            # Вычисляем количество узлов для удаления
            num_to_remove = int(original_size * x / 100)

            # Случайно выбираем узлы только из LCC
            nodes_to_remove = sample(list(lcc_nodes), num_to_remove)

            # Удаляем узлы из LCC
            for node in nodes_to_remove:
                # Удаляем связи с соседями
                for neighbor in current_lcc.get(node, set()):
                    current_lcc[neighbor].discard(node)
                del current_lcc[node]

            # Находим новые компоненты в измененной LCC
            new_components = get_weakly_connected_components(current_lcc)

            if new_components:
                new_lcc_size = len(max(new_components, key=len))
                remaining_nodes = sum(len(comp) for comp in new_components)
                results[x].append(new_lcc_size / remaining_nodes if remaining_nodes > 0 else 0)

    # Усредняем результаты
    return {x: sum(results[x]) / len(results[x]) for x in results}


def get_weakly_connected_components(graph_dict):
    visited = set()
    components = []

    for node in graph_dict:
        if node not in visited:
            # BFS для нахождения компоненты
            queue = [node]
            visited.add(node)
            component = []

            while queue:
                current = queue.pop(0)
                component.append(current)

                for neighbor in graph_dict.get(current, set()):
                    if neighbor not in visited:
                        visited.add(neighbor)
                        queue.append(neighbor)

            components.append(component)

    return components


In [ ]:
x_values = [5, 10, 15, 20, 25, 30]
resilience_results = analyze_component_resilience(graph, x_values)

# Визуализация результатов
print("Устойчивость наибольшей компоненты:")
print("Процент удаления | Доля вершин в LCC")
print("-------------------------------")
for x in sorted(resilience_results):
    print(f"{x:15}% | {resilience_results[x]:.4f}")

### _Часть 2_

In [ ]:
def analyze_targeted_removal(graph, x_values):
    results = {}
    components = dfs_stack(graph)
    lcc = subgraph(components, graph)
    original_size = len(lcc)

    # Предварительно сортируем узлы по степени (по убыванию)
    sorted_nodes = sorted(lcc.items(),
                         key=lambda item: len(item[1]),
                         reverse=True)

    for x in x_values:
        # Создаем копию только для узлов LCC
        current_lcc = {node: set(neighbors) for node, neighbors in lcc.items()}

        # Вычисляем количество узлов для удаления
        num_to_remove = int(original_size * x / 100)

        # Выбираем топ-N узлов с наибольшей степенью
        nodes_to_remove = [node for node, _ in sorted_nodes[:num_to_remove]]

        # Удаляем узлы из LCC
        for node in nodes_to_remove:
            # Удаляем связи с соседями
            for neighbor in current_lcc.get(node, set()):
                current_lcc[neighbor].discard(node)
            del current_lcc[node]

        # Находим новые компоненты в измененной LCC
        new_components = get_weakly_connected_components(current_lcc)

        if new_components:
            new_lcc_size = len(max(new_components, key=len))
            remaining_nodes = sum(len(comp) for comp in new_components)
            results[x] = new_lcc_size / remaining_nodes if remaining_nodes > 0 else 0

    return results


In [ ]:
x_values = [5, 10, 15, 20, 25, 30]
resilience_results = analyze_targeted_removal(graph, x_values)

# Визуализация результатов
print("Устойчивость наибольшей компоненты:")
print("Процент удаления | Доля вершин в LCC")
print("-------------------------------")
for x in sorted(resilience_results):
    print(f"{x:15}% | {resilience_results[x]:.4f}")

In [ ]:
def process_directed_graph_file(path, type='txt'):
    graph = defaultdict(set)

    unique_edges = 0
    all_edges = 0

    try:
        with open(path, 'r') as file:
            if type == 'txt':
                for line in file:
                    line = line.strip()

                    if line.startswith('#') or not line:
                        continue

                    u, v = line.split()
                    u = int(u)
                    v = int(v)


                    all_edges += 1

                    if u not in graph[v]:
                        graph[u].add(v)
                        unique_edges += 1
            elif type == 'mtx':
                lines = file.readlines()

                head = lines.pop(0)
                symmetric = 'symmetric' in head

                _, _, all_edges = [int(i) for i in lines.pop(0).split()]

                for line in lines:
                    line = line.strip()

                    u, v = line.split()
                    u = int(u)
                    v = int(v)

                    if v not in graph[u]:
                        graph[u].add(v)
                        if symmetric:
                            graph[v].add(u)

                        unique_edges += 1
            elif type == 'csv':
                lines = file.readlines()
                lines.pop(0)
                for line in lines:                    
                    data = list(map(int, line.strip().split(',')))
                    u = data[0]
                    v = data[1]
                    all_edges += 1

                    if v not in graph[u]:
                        graph[u].add(v)

                        unique_edges += 1

    except FileNotFoundError:
        print(f"ошибка: файл {path} не найден.")
        return None

    except Exception as e:
        print(f"произошла ошибка при обработке файла: {e}")
        return None

    return dict(graph), unique_edges, all_edges


In [ ]:
file = "./Wiki-Vote.txt"
graph, edges, _ = make_graph(file)

In [ ]:
print(edges)

In [ ]:
def count_scc(graph):
    # Первый проход DFS для определения порядка завершения
    visited = set()
    order = []

    for u in graph:
        if u not in visited:
            stack = [(u, False)]

            while stack:
                cur, processed = stack.pop()
                if processed:
                    order.append(cur)
                    continue
                if cur in visited:
                    continue

                if cur not in visited:
                    visited.add(cur)
                    stack.append((cur, True))
                    for v in graph.get(cur, set()):
                        if v not in visited:
                            stack.append((v, False))

    # Инвертируем граф
    reversed_graph = defaultdict(set)
    for src in graph:
        for dst in graph[src]:
            reversed_graph[dst].add(src)

    # Второй проход DFS в обратном порядке по инвертированному графу
    visited = set()
    components = []

    for u in reversed(order):
        if u not in visited:
            stack = [u]
            visited.add(u)
            component = []

            while stack:
                cur = stack.pop()
                component.append(cur)

                for v in reversed_graph.get(cur, set()):
                    if v not in visited:
                        stack.append(v)
                        visited.add(v)

            components.append(component)
    return components

In [ ]:
scc = count_scc(graph)

print(len(scc))

# Вычисление расстояний между вершинами сети

Реализуем возможность находить расстояние между двумя произвольными вершинами графа с помощью базового алгоритма и его модификации. Для базового алгоритма выбрем несколько ориентиров (landmarks), для каждого из них произведем поиск в ширину и запишем расстояние до каждой вершины графа. Полученные деревья кратчайших путей (shortest path trees) будем использовать для аппроксимации расстояния между двумя вершинами. Будем сравнивать его с точным расстоянием, полученным по алгоритму Дейкстры.

Перепишем код поиска в ширину для построения дерева кратчайших путей:

In [ ]:
def spt(graph, u):
    tree = {u: 0}
    queue = deque([u])

    while queue:
        cur = queue.popleft()

        for v in graph.get(cur, set()):
            if v not in tree:
                tree[v] = tree[cur] + 1
                queue.append(v)

    return tree

Напишем функцию для выбора ориентиров в графе. Реализуем следующие алгоритмы выбора:
1. Степень центральности (degree centrality) - вершины с наибольшей степенью
2. Случайные вершины
3. Степень близости (closeness centrality) - вершины с наибольшим числом кратчайших путей, пролегающих верез них

Последний критерий является самым многообещающим, но самым дорогим для вычисления. Чтобы не вычислять всевозможные кратчайшие пути в графе (сложность O(nm)), произведем аппроксимацию: выберем несколько случайных пар вершин, проложим путь между ними и выберем те вершины, которые попадут в наибольшее число путей.

Еще раз модифицирем BFS так, чтобы он возвращал путь до искомой вершины:

In [ ]:
def get_path(graph, start, goal):
    visited = {start}
    queue = deque([[start]])

    while queue:
        cur = queue.popleft()
        last = cur[-1]
        for v in graph.get(last, set()):
            if v == goal:
                return cur + [v]
            if v not in visited:
                visited.add(v)
                queue.append(cur + [v])
    return []

Теперь напишем функцию для выбора ориентиров и построения деревьев для них. Добавим возможность выбора режима работы и числа искомых ориентиров:

In [ ]:
def get_landmarks(graph, n=16, mode='degree'):
    shortest_path_trees = dict()
    landmarks = []

    if mode == 'degree':
        high_degree_nodes = [node for node, _ in sorted(graph.items(), key=lambda item: len(item[1]), reverse=True)[:n]]
        landmarks.extend(high_degree_nodes)
    elif mode == 'random':
        while n:
            node = choice(list(graph.keys()))
            if node not in landmarks:
                landmarks.append(node)
                n -= 1
    elif mode == 'best-coverage':
        coverage = dict()
        for _ in range(n * 50):
            start, goal = sample(list(graph.keys()), 2)
            path = get_path(graph, start, [goal])
            for node in path:
                if node in coverage:
                    coverage[node] += 1
                else:
                    coverage[node] = 1
        nodes = [(i, coverage[i]) for i in coverage]
        nodes.sort(key=lambda x: x[1], reverse=True)
        landmarks = [i[0] for i in nodes[:n]]

    for node in landmarks:
        shortest_path_trees[node] = spt(graph, node)

    return shortest_path_trees